In [0]:
%run ./00_config

# Scratch / Practice Notebook

This notebook is my personal scratch space for experimenting and practicing new PySpark/Spark SQL concepts while building this project.

**It is NOT part of the graded pipeline.** The real pipeline is: `00_config` -> `01_setup_landing_zone` -> `02_bronze_ingestion` -> `03_silver_cleaning` -> `04_gold_aggregations` -> `05_analytics` -> `06_incremental_test`.

# 00 · Config
All project names and paths in one place. Every other notebook loads this with `%run ./00_config`.

✅ Config loaded. Landing folder: /Volumes/workspace/nyc_taxi/pipeline_files/landing


In [0]:
demand = spark.table(GOLD_DEMAND_TABLE)
display(demand)

day_type,pickup_hour,total_trips,num_days,avg_trips_per_day
Weekday,0,793113,152,5218.0
Weekday,1,462301,152,3041.0
Weekday,2,279110,152,1836.0
Weekday,3,186638,152,1228.0
Weekday,4,194068,152,1277.0
Weekday,5,362331,152,2384.0
Weekday,6,934881,152,6151.0
Weekday,7,1637277,152,10772.0
Weekday,8,1993923,152,13118.0
Weekday,9,1877022,152,12349.0


Databricks visualization. Run in Databricks to view.

### Q1 - When is demand highest?

Weekday demand has two clear peaks: one in the morning (around 8am, the work commute) and a much bigger one in the evening (around 6pm, people heading home). Weekend demand has one broad, smooth peak building through the afternoon into the evening, with no sharp commute spikes, since people are not travelling to and from work. This suggests the business should plan for extra driver supply at both weekday rush hours, and a steadier, later supply on weekends.

In [0]:
payment = spark.table(GOLD_PAYMENT_TABLE)
display(payment)

payment_method,total_trips,avg_fare,avg_tip,avg_tip_percent,pct_trips_with_tip,share_of_trips_pct
Credit card,36335438,13.24,2.98,25.44,96.0,72.43
Cash,13557412,12.32,0.0,0.0,0.0,27.02
No charge,184425,21.49,0.0,0.01,0.1,0.37
Dispute,58354,14.18,0.0,0.03,0.1,0.12
Unknown,33546,36.5,0.06,0.1,0.5,0.07


Databricks visualization. Run in Databricks to view.

### Q2 - Does payment method affect tipping?

Credit card riders tip about 25% on average, while cash, no-charge, and dispute trips all show close to 0%. However, this does not mean cash riders never tip in real life - cash tips are handed directly to the driver and are never recorded in the system, while card tips are automatically logged. The real takeaway is that our tipping data is only reliable for card payments, not a true measure of who tips more.

In [0]:
vendor = spark.table(GOLD_VENDOR_TABLE)
display(vendor)

vendor_name,total_trips,avg_fare,avg_distance_miles,avg_speed_mph,avg_tip_percent,share_of_trips_pct
VeriFone,31222947,13.21,3.09,12.03,18.09,62.24
Unknown vendor,298660,15.02,3.62,11.76,15.95,0.6
Creative Mobile Technologies,18647568,12.72,2.85,14.66,19.02,37.17


Databricks visualization. Run in Databricks to view.

### Q3 - Which vendor performs better?

VeriFone handles the most trips (about 61% of all rides), followed by Creative Mobile Technologies (about 38%), with a small "Unknown vendor" group making up under 1%. Average fare and tip percent are close between the two main vendors, so neither is clearly "better" for riders. Differences like average speed likely reflect the routes and areas each vendor's cars happened to serve, rather than real differences in driving performance.